# Alternative tool paths on T3 and T4

Both Claude models score the lowest tool selection accuracy of the main agents on
template T3 (ranking) and T4 (cross-segment comparison), yet score 4 or 5 on insight
in every one of those runs. This notebook reads their traces, compares the tool path
they took with the gold path, and shows why the shorter path still yields the right
answer.

## Findings

- **Every low-recall Claude run on T3 and T4 scored insight >= 4.** Across 5 seeds, 4 valid T3 tasks and 5 T4 tasks, no run that missed a gold tool produced a wrong answer. Low recall here is a property of the gold path, not of the reasoning.
- **T3: the agents skip `rank`.** Gold is `filter -> summarise -> rank(top_k=3)`. Both Claude models call `filter -> summarise` and stop, because `summarise` already returns the aspects sorted by total count in descending order. The top 3 are its first three rows. Recall is 2/3 for an answer identical to gold.
- **T4: the agents skip `summarise` and use `ztest` instead.** Gold is one `filter` over both industries then `summarise(group_by='industry')`. Both Claude models filter each industry separately and call `ztest` on the pair. The `ztest` result carries both negative rates, both counts, the gap and the p-value, a strict superset of what `summarise` gives. Recall is 1/2 for an answer that matches gold and adds a significance test.
- **The other main models take the same shortcuts but also call the gold tool.** GPT-5.6 and DeepSeek call `rank` on T3 and both `summarise` and `ztest` on T4, so their tool sets are supersets of gold and score full recall for the same answer. The Claude models are the only ones penalised because they are the only ones that consistently take the leaner path.
- **The metric has no notion of equivalence.** `tool_selection_accuracy` is recall of the gold tool set (`scoring.py`, `evaluation_metrics`). Any path that reaches the answer through a different tool is penalised, even when its output subsumes the gold tool's output.
- **Two hard T4 gold paths look inconsistent.** T4-H2 is recorded as `filter -> summarise -> filter` and T4-H3 as two bare `filter` calls with no `summarise`. On the same strategy the Claude models score 0.8 to 1.0 recall there versus 0.5 on the other T4 tasks. Those two gold rows are worth a check in the gold-path notebook.

## Load the Claude T3 and T4 traces

Each trace is re-scored with the repo's `score_run` so the agent path, gold path and answers sit in one row, then joined with the judge's insight score from the run's `scores.csv`. The pruned task T3-H3 is dropped because it has no score.

In [1]:
import glob, json, os, sys, textwrap
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
from cx_agent_bench.scoring import read_trace, score_run
from cx_agent_bench.tasks_gold import load_gold_tasks

pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 200)

gold = load_gold_tasks()
gold = {t["task_id"]: t for t in gold} if isinstance(gold, list) else gold

rows = []
for d in sorted(glob.glob(str(REPO / "traces" / "chattermill" / "claude-*"))):
    if not os.path.exists(f"{d}/scores.csv"):
        continue  # aborted run, never scored
    sc = pd.read_csv(f"{d}/scores.csv").set_index("task_id")
    for f in glob.glob(f"{d}/*T[34]-*.jsonl"):
        r = score_run(f, gold)
        if r["task_id"] not in sc.index:
            continue  # pruned task
        r["insight"] = sc.loc[r["task_id"], "insight_score"]
        r["tsa"] = sc.loc[r["task_id"], "tool_selection_accuracy"]
        r["run"] = os.path.basename(d)
        rows.append(r)
runs = pd.DataFrame(rows)
runs["model"] = runs["run"].str.extract(r"(claude-[a-z]+-5)")
runs["seed"] = runs["run"].str.extract(r"_s(\d+)")
print(len(runs), "runs:", runs["model"].value_counts().to_dict())

90 runs: {'claude-fable-5': 45, 'claude-sonnet-5': 45}


## Recall vs insight per task

Mean over the 5 seeds. `path_match` is exact equality with the gold path.

In [2]:
summary = (runs.groupby(["model", "task_id"])[["tsa", "insight", "path_match"]]
           .mean().round(2))
summary.columns = ["tool_selection_accuracy", "insight_score", "path_match"]
print(summary.to_string())

low = runs[(runs["tsa"] < 1)]
print(f"\nlow-recall runs: {len(low)}; of those with insight >= 4: {(low['insight'] >= 4).sum()}")

                         tool_selection_accuracy  insight_score  path_match
model           task_id                                                    
claude-fable-5  T3-E1                       0.67            5.0         0.0
                T3-E2                       0.67            5.0         0.0
                T3-H1                       0.67            5.0         0.0
                T3-H2                       0.67            5.0         0.0
                T4-E1                       0.50            5.0         0.0
                T4-E2                       0.50            5.0         0.0
                T4-H1                       0.50            4.2         0.0
                T4-H2                       0.80            5.0         0.0
                T4-H3                       1.00            5.0         0.0
claude-sonnet-5 T3-E1                       0.67            5.0         0.0
                T3-E2                       0.67            5.0         0.0
            

## Gold paths for the tasks

In [3]:
for t in sorted(runs["task_id"].unique()):
    g = gold[t]
    print(f"{t} | {g['question']}")
    for call in json.loads(g["gold_tool_path_json"]):
        print("    ", call["tool"], json.dumps(call["args"]))
    print()

T3-E1 | What are the top 3 topics with the most number of complaints in Travel Booking?
     filter {"industry": ["Travel Booking"], "sentiment": ["negative"]}
     summarise {"group_by": "aspect", "ref": {"args": {"industry": ["Travel Booking"], "sentiment": ["negative"]}, "tool": "filter"}}
     rank {"by": "total_count", "ref": {"args": {"group_by": "aspect", "ref": {"args": {"industry": ["Travel Booking"], "sentiment": ["negative"]}, "tool": "filter"}}, "tool": "summarise"}}

T3-E2 | What are the top 3 topics with the most number of complaints in Groceries?
     filter {"industry": ["Groceries"], "sentiment": ["negative"]}
     summarise {"group_by": "aspect", "ref": {"args": {"industry": ["Groceries"], "sentiment": ["negative"]}, "tool": "filter"}}
     rank {"by": "total_count", "ref": {"args": {"group_by": "aspect", "ref": {"args": {"industry": ["Groceries"], "sentiment": ["negative"]}, "tool": "filter"}}, "tool": "summarise"}}

T3-H1 | What are the top 3 topics with the most nu

## Agent paths vs gold on the low-recall runs

One block per (task, model): the distinct tool sequences the agent used across seeds, and which gold tool each is missing.

In [4]:
def tools(path_json):
    return [c["tool"] for c in json.loads(path_json)]

for (task, model), g in low.groupby(["task_id", "model"]):
    gold_tools = set(tools(g.iloc[0]["gold_tool_path"]))
    print(f"##### {task} | {model} | {len(g)} low-recall runs | insight = {sorted(g['insight'].tolist())}")
    for path, gg in g.groupby("agent_tool_path"):
        used = tools(path)
        print(f"  agent path {used}  (x{len(gg)} seeds)  missing gold tool: {sorted(gold_tools - set(used))}")
        for call in json.loads(path):
            args = {k: v for k, v in call["args"].items() if k not in ("ref", "ref_a", "ref_b")}
            print("      ", call["tool"], json.dumps(args))
    print()

##### T3-E1 | claude-fable-5 | 5 low-recall runs | insight = [5, 5, 5, 5, 5]
  agent path ['filter', 'summarise']  (x5 seeds)  missing gold tool: ['rank']
       filter {"industry": ["Travel Booking"], "sentiment": ["negative"]}
       summarise {"group_by": "aspect"}

##### T3-E1 | claude-sonnet-5 | 5 low-recall runs | insight = [5, 5, 5, 5, 5]
  agent path ['filter', 'summarise']  (x5 seeds)  missing gold tool: ['rank']
       filter {"industry": ["Travel Booking"], "sentiment": ["negative"]}
       summarise {"group_by": "aspect", "min_n": 30}

##### T3-E2 | claude-fable-5 | 5 low-recall runs | insight = [5, 5, 5, 5, 5]
  agent path ['filter', 'summarise']  (x5 seeds)  missing gold tool: ['rank']
       filter {"industry": ["Groceries"], "sentiment": ["negative"]}
       summarise {"group_by": "aspect"}

##### T3-E2 | claude-sonnet-5 | 5 low-recall runs | insight = [5, 5, 5, 5, 5]
  agent path ['filter', 'summarise']  (x3 seeds)  missing gold tool: ['rank']
       filter {"industry"

## Why the shorter path is enough: the tool outputs

From Fable seed 2555. The `summarise` result is already sorted by `total_count` descending, so `rank` adds nothing for a top-3 question. The `ztest` result carries both groups' negative rates and counts, so `summarise` adds nothing for a two-way comparison.

In [5]:
def tool_results(run_dir, task, tool_names):
    f = glob.glob(str(REPO / "traces" / "chattermill" / run_dir / f"*__{task}__*.jsonl"))[0]
    _, steps, _ = read_trace(f)
    return [s for s in steps if s.get("tool") in tool_names]

FABLE_S2555 = "claude-fable-5_20260904_064122_s2555"

print("=== T3-E1: summarise output (first 4 of the rows) ===")
s = tool_results(FABLE_S2555, "T3-E1", {"summarise"})[0]
res = s.get("result") or s.get("tool_result") or s.get("output")
print(json.dumps({k: v for k, v in res.items() if k != "rows"}))
print(pd.DataFrame(res["rows"])[["group", "total_count", "neg_count", "prevalence"]].head(4).to_string(index=False))

print("\n=== T4-E1: ztest output ===")
s = tool_results(FABLE_S2555, "T4-E1", {"ztest"})[0]
res = s.get("result") or s.get("tool_result") or s.get("output")
print(json.dumps(res, indent=1))

=== T3-E1: summarise output (first 4 of the rows) ===
{"status": "ok", "id": "t1", "selection_rows": 1145}
         group  total_count  neg_count  prevalence
   app-website          215        215      0.1878
account-access          149        149      0.1301
         email          138        138      0.1205
         phone          116        116      0.1013

=== T4-E1: ztest output ===
{
 "status": "ok",
 "group_a": {
  "total": 160,
  "neg": 32,
  "non_neg": 128,
  "neg_rate": 0.2
 },
 "group_b": {
  "total": 120,
  "neg": 91,
  "non_neg": 29,
  "neg_rate": 0.7583
 },
 "sufficient": true,
 "gap_pp": -55.8,
 "z": -9.316,
 "p_value": 0.0,
 "significant": true
}


## Example answers against gold

One T3 and one T4 example per model, all from seed 2555, all scored 5 by the judge.

In [6]:
def show(task, model, seed="2555"):
    r = runs[(runs.task_id == task) & (runs.model == model) & (runs.seed == seed)].iloc[0]
    print(f"===== {task} | {model} | seed {seed} | recall {r['tsa']} | insight {r['insight']} =====")
    print("Q:    ", r["question"])
    print("path: ", " -> ".join(tools(r["agent_tool_path"])), "   (gold:", " -> ".join(tools(r["gold_tool_path"])) + ")")
    print("GOLD: ", textwrap.fill(str(r["gold_answer"]), 110, subsequent_indent="       "))
    print("AGENT:", textwrap.fill(str(r["agent_answer"]), 110, subsequent_indent="       "))
    print()

for model in ["claude-fable-5", "claude-sonnet-5"]:
    show("T3-E1", model)
    show("T4-E1", model)

===== T3-E1 | claude-fable-5 | seed 2555 | recall 0.667 | insight 5 =====
Q:     What are the top 3 topics with the most number of complaints in Travel Booking?
path:  filter -> summarise    (gold: filter -> summarise -> rank)
GOLD:  The top 3 topics with the most complaints in Travel Booking are: the app or website (215), account access
       (149) and email support (138).
AGENT: The top 3 topics with the most complaints (negative reviews) in Travel Booking are: app or website (215
       complaints), account access (149 complaints) and email (138 complaints). Together these account for 502
       of the 1,145 total complaints in the industry (43.8%).

===== T4-E1 | claude-fable-5 | seed 2555 | recall 0.5 | insight 5 =====
Q:     Looking only at reviews about price and value for money, is the negative sentiment rate higher in Banking or Ride Hailing?
path:  filter -> filter -> ztest    (gold: filter -> summarise)
GOLD:  Ride Hailing has the higher negative sentiment rate (75.8%) for 

## How the other main models covered the gold tools

Tool sets used on the same tasks, all seeds. GPT-5.6 and DeepSeek reach `rank` on T3 and both `summarise` and `ztest` on T4, so their sets are supersets of gold.

In [7]:
rows = []
for d in sorted(glob.glob(str(REPO / "traces" / "chattermill" / "*_s*"))):
    if not os.path.exists(f"{d}/scores.csv"):
        continue
    model = os.path.basename(d).split("_2026")[0]
    for f in glob.glob(f"{d}/*T[34]-*.jsonl"):
        start, steps, _ = read_trace(f)
        if start["task_id"] == "T3-H3":
            continue
        used = " + ".join(sorted({s["tool"] for s in steps
                                  if s.get("status") == "ok" and s.get("tool") != "answer"}))
        rows.append((start["task_id"][:2], model, used))
tool_sets = pd.DataFrame(rows, columns=["template", "model", "tool_set"])
print(tool_sets.groupby(["template", "model"])["tool_set"]
      .value_counts().unstack(fill_value=0).to_string())

tool_set                     filter  filter + rank + summarise  filter + summarise  filter + summarise + ztest  filter + ztest
template model                                                                                                                
T3       claude-fable-5   0       0                          0                  20                           0               0
         claude-sonnet-5  0       0                          2                  18                           0               0
         deepseek-v3.2    1       0                         19                   0                           0               0
         gpt-5.6-sol      0       0                         20                   0                           0               0
         gpt-5.6-terra    0       0                         20                   0                           0               0
T4       claude-fable-5   0       3                          0                   7                           0 

## Examples for the thesis table

Representative runs from seed 2555 for Section 4.3.4, printed as a plain table and as Markdown.

In [8]:
# Representative examples for the thesis table (Section 4.3.4), all seed 2555.
# Paths, recall and insight come from the traces; the last column is a written
# comparison of the agent answer with the gold answer.
EXAMPLES = [
    ("T3-E1", "claude-fable-5",  "Same three topics and counts, adds their share of all complaints"),
    ("T3-H1", "claude-sonnet-5", "Same three topics and counts"),
    ("T4-E1", "claude-fable-5",  "Same two rates, adds counts and a significant z-test"),
    ("T4-E1", "gpt-5.6-sol",     "Same answer as the Claude run; full recall because summarise was also called"),
    ("T4-H1", "claude-fable-5",  "Same direction and rates; judge scored 4, the one T4 case where the shortcut was not a free pass"),
]

def example_row(task, agent, note, seed="2555"):
    d = [p for p in glob.glob(str(REPO / "traces" / "chattermill" / f"{agent}_*_s{seed}"))
         if os.path.exists(f"{p}/scores.csv")][0]
    f = glob.glob(f"{d}/*__{task}__*.jsonl")[0]
    r = score_run(f, gold)
    sc = pd.read_csv(f"{d}/scores.csv").set_index("task_id")
    return {
        "task_id": task,
        "question": r["question"],
        "Agent": agent,
        "Gold path": ", ".join(tools(r["gold_tool_path"])),
        "Agent path": ", ".join(tools(r["agent_tool_path"])),
        "Tool Selection Accuracy": sc.loc[task, "tool_selection_accuracy"],
        "Insight Score": int(sc.loc[task, "insight_score"]),
        "Gold answer": r["gold_answer"],
        "Agent answer": r["agent_answer"],
        "Answer vs gold": note,
    }

examples = pd.DataFrame([example_row(*e) for e in EXAMPLES])
pd.set_option("display.max_colwidth", None)
print(examples.to_string(index=False))

# # Markdown version for pasting into the thesis
# cols = list(examples.columns)
# md = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
# md += ["| " + " | ".join(str(v) for v in row) + " |" for row in examples.itertuples(index=False)]
# print(chr(10) + chr(10).join(md))

task_id                                                                                                                   question           Agent               Gold path                                  Agent path  Tool Selection Accuracy  Insight Score                                                                                                                              Gold answer                                                                                                                                                                                                                                                                                                                                                                                        Agent answer                                                                                   Answer vs gold
  T3-E1                                            What are the top 3 topics with the most number of complaints in Travel B

## Chosen examples

The two rows used in the thesis table for Section 4.3.4: T3-E1 and T4-E1, both from claude-fable-5 at seed 2555.

In [9]:
# The two examples chosen for the thesis: one T3 and one T4 shortcut, both
# claude-fable-5 at seed 2555, the shorter answers available in the table above.
chosen = examples[(examples["Agent"] == "claude-fable-5")
                  & (examples["task_id"].isin(["T3-E1", "T4-E1"]))].reset_index(drop=True)
print(chosen.to_string(index=False))
chosen

task_id                                                                                                                   question          Agent               Gold path            Agent path  Tool Selection Accuracy  Insight Score                                                                                                                              Gold answer                                                                                                                                                                                                                                                                                                                    Agent answer                                                   Answer vs gold
  T3-E1                                            What are the top 3 topics with the most number of complaints in Travel Booking? claude-fable-5 filter, summarise, rank     filter, summarise                    0.667              5 The top 3 topi

,task_id,question,Agent,Gold path,Agent path,Tool Selection Accuracy,Insight Score,Gold answer,Agent answer,Answer vs gold
0,T3-E1,What are the top 3 topics with the most number of complaints in Travel Booking?,claude-fable-5,"filter, summarise, rank","filter, summarise",0.667,5,"The top 3 topics with the most complaints in Travel Booking are: the app or website (215), account access (149) and email support (138).","The top 3 topics with the most complaints (negative reviews) in Travel Booking are: app or website (215 complaints), account access (149 complaints) and email (138 complaints). Together these account for 502 of the 1,145 total complaints in the industry (43.8%).","Same three topics and counts, adds their share of all complaints"
1,T4-E1,"Looking only at reviews about price and value for money, is the negative sentiment rate higher in Banking or Ride Hailing?",claude-fable-5,"filter, summarise","filter, filter, ztest",0.500,5,"Ride Hailing has the higher negative sentiment rate (75.8%) for price and value for money, compared to 20.0% for Banking.","For price and value-for-money reviews, the negative sentiment rate is much higher in Ride Hailing than in Banking. Ride Hailing: 75.8% negative (91 of 120 mentions); Banking: 20.0% negative (32 of 160 mentions). The gap of 55.8 percentage points is statistically significant (two-proportion z-test, z = 9.3, p < 0.001).","Same two rates, adds counts and a significant z-test"
